# Attribute Access & State Management

### 1. Instance Attributes vs. Class Attributes

Instance Attributes (The Unique State)
We just mastered these. Instance attributes are owned by a specific instance of the class. They are usually created inside __init__ using self.attribute_name.

Rule: If the data should be unique to each object (like a user's ID, a student's name, or a database connection's URL), it must be an instance attribute.

Class Attributes (The Shared State)
Class attributes are variables defined directly under the class statement, outside of any method (including __init__). They are owned by the class itself, not by any specific instance.

Rule: If the data should be exactly the same for every object created from this blueprint, or if you need to track data across all objects, it should be a class attribute.

Let's look at how they work together:

In [43]:
class SoftwareEngineer:
    # --- CLASS ATTRIBUTES ---
    # These belong to the SoftwareEngineer blueprint.
    # Shared by ALL engineers.
    company = "Tekion Corp"
    minimum_salary = 100000
    total_engineers_hired = 0  # We can use this as a counter!

    def __init__(self, name, primary_language):
        # --- INSTANCE ATTRIBUTES ---
        # These belong to the specific engineer being created.
        self.name = name
        self.primary_language = primary_language
        
        # When a new engineer is initialized, we update the CLASS attribute
        SoftwareEngineer.total_engineers_hired += 1

# Let's create two distinct instances
engineer_1 = SoftwareEngineer("Shubham", "cpp")
engineer_2 = SoftwareEngineer("Alice", "Python")

# Accessing Instance Attributes
print(f"{engineer_1.name} writes {engineer_1.primary_language}") # Shubham writes Java
print(f"{engineer_2.name} writes {engineer_2.primary_language}") # Alice writes Python

# Accessing Class Attributes
# Best practice: Access class attributes using the ClassName
print(f"Company: {SoftwareEngineer.company}") # Tekion Corp
print(f"Total Hired: {SoftwareEngineer.total_engineers_hired}") # 2

Shubham writes cpp
Alice writes Python
Company: Tekion Corp
Total Hired: 2


### 2. The Tricky Part: Attribute Resolution Order
You can actually access a class attribute through an instance (e.g., engineer_1.company). But you must be very careful when modifying them.

When you ask an instance for an attribute (engineer_1.company), Python follows a specific search order:

Step 1: It checks the instance's own namespace (the instance's __dict__). Does engineer_1 have an attribute named company?

Step 2: If it doesn't find it there, it falls back to the class namespace. Does SoftwareEngineer have a class attribute named company?

Step 3: If it's not there, it checks parent classes (which we'll cover in inheritance), and finally throws an AttributeError if it can't find it anywhere.

In [44]:
class SystemConfig:
    timeout_seconds = 30 # Class attribute

config_a = SystemConfig()
config_b = SystemConfig()

# Both instances fall back to the class attribute
print(config_a.timeout_seconds) # 30
print(config_b.timeout_seconds) # 30

# DANGER: What happens if we do this?
config_a.timeout_seconds = 60

print(config_a.timeout_seconds) # 60 -> Expected.
print(config_b.timeout_seconds) # 30 -> Wait, what?
print(SystemConfig.timeout_seconds) # 30 -> The class attribute didn't change!

30
30
60
30
30


```text
Why did this happen?
When we wrote config_a.timeout_seconds = 60, Python did not change the class attribute. Instead, it created a brand-new instance attribute named timeout_seconds directly on config_a.

Now, when config_a looks for timeout_seconds, it finds its own instance attribute (Step 1) and stops looking. config_b doesn't have an instance attribute, so it still falls back to the class attribute (Step 2).

The Golden Rule of Mutation:

To read a class attribute, you can use the instance (self.attr) or the class (Class.attr).

To modify a class attribute, you must use the class name (Class.attr = new_value).

### 3. Method Types: Standard, @classmethod, and @staticmethod
Just as we have instance and class variables, we have different types of methods to interact with them.

1) Instance Methods (Standard):

Signature: def method(self, arg):

Usage: They take self as the first argument. They can access both instance attributes (via self) and class attributes (via self.__class__ or the class name). These are your bread and butter.

2) Class Methods (@classmethod):

Signature: def method(cls, arg):

Usage: They take cls (the class itself) as the first argument instead of self. They cannot access instance attributes because they don't know about any specific instance. They are used to modify class state, or very commonly, as Alternative Constructors.

In [45]:
import datetime

class User:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    # A class method takes 'cls' instead of 'self'
    @classmethod
    def from_birth_year(cls, name, birth_year):
        """Alternative constructor: calculates age from birth year."""
        current_year = datetime.date.today().year
        age = current_year - birth_year
        # 'cls' is the User class. This returns a new User instance!
        return cls(name, age)

# Creating a user using the standard __init__
user1 = User("Alice", 25)

# Creating a user using our custom class method constructor
user2 = User.from_birth_year("Bob", 1999)

3) Static Methods (@staticmethod):

Signature: def method(arg):

Usage: They take neither self nor cls. They are basically regular functions that happen to live inside a class's namespace because they logically belong there. They cannot modify object state or class state.

In [46]:
class MathUtils:
    @staticmethod
    def is_even(number):
        # No 'self', no 'cls'. Just a plain function bundled in the class.
        return number % 2 == 0

print(MathUtils.is_even(10)) # True

True


## Encapsulation & Properties

### 1. "Protected" Attributes (The Single Underscore)
If you want to signal to other developers that an attribute or method is for internal use only and shouldn't be touched from outside the class, you prefix it with a single underscore _.

In [47]:
class DatabaseConnection:
    def __init__(self, host):
        self.host = host
        # The underscore tells other devs: "This is internal state. Don't touch it."
        self._is_connected = False 

    def connect(self):
        # Internal methods can modify it safely
        self._is_connected = True

db = DatabaseConnection("localhost")

# WARNING: Python WILL let you do this, but you are breaking the social contract.
# Your code reviewer will flag this!
db._is_connected = True

### 2. "Private" Attributes (Name Mangling with Double Underscore)
If you really want to make it difficult for someone to accidentally overwrite an internal attribute (especially when dealing with inheritance, which we'll see later), you use a double underscore __.

This triggers a mechanism called Name Mangling.

In [48]:
class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner
        # Double underscore triggers name mangling
        self.__balance = balance 

    def deposit(self, amount):
        if amount > 0:
            self.__balance += amount

    def get_balance(self):
        return self.__balance

account = BankAccount("Shubham", 1000)

# This works fine, it uses the public method to access the internal state
print(account.get_balance()) # 1000

# This will fail!
# print(account.__balance) # AttributeError: 'BankAccount' object has no attribute '__balance'

1000


```text
It looks like the variable is truly private, right? Not quite. Python just changed its name behind your back.

When Python sees __balance inside the BankAccount class, it silently renames it to _BankAccount__balance to prevent accidental name collisions in subclasses.

In [49]:
# You can still access it if you know the secret name!
# (Again, never do this in real code unless debugging)
print(account._BankAccount__balance) # 1000

1000


### 3. The Pythonic Way: The @property Decorator

In Java, it's standard practice to make every variable private and write a get_variable() and set_variable() method.

Do not do this in Python. It is considered un-pythonic because it litters your code with boilerplate getter and setter methods.

Python's philosophy is: start with simple public attributes. If, later on, you realize you need to add validation, calculation, or side-effects when an attribute is accessed or changed, you use the @property decorator.

@property allows you to write methods that look and act exactly like simple attributes from the outside.

In [50]:
class TemperatureSensor:
    def __init__(self, initial_celsius):
        # We store the actual value in a "protected" attribute
        self._celsius = initial_celsius

    # 1. THE GETTER
    # This turns the method celsius() into a read-only attribute.
    @property
    def celsius(self):
        print("Fetching temperature from hardware...")
        return self._celsius

    # 2. THE SETTER
    # This allows us to validate data before assignment.
    # The decorator name must match the property name exactly: @property_name.setter
    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("Temperature below absolute zero is impossible!")
        print(f"Setting temperature to {value}")
        self._celsius = value

# Look how clean the usage is!
sensor = TemperatureSensor(25)

# It looks like we are accessing a simple variable, but it's calling the GETTER method under the hood.
current_temp = sensor.celsius 
# Output: Fetching temperature from hardware...

# It looks like simple assignment, but it's calling the SETTER method under the hood.
sensor.celsius = 30 
# Output: Setting temperature to 30

# The setter's validation protects our internal state.
# sensor.celsius = -300 
# Output: ValueError: Temperature below absolute zero is impossible!

Fetching temperature from hardware...
Setting temperature to 30
